# 03 — Chronos-Bolt inference (Amazon)

Template to run **Chronos-Bolt small** (`amazon/chronos-bolt-small`) over all anonymized series.

**Isolated environment:** `.venv_chronos` (see `requirements-chronos.txt`).

**Reason for isolation:** `chronos-forecasting` has dependencies incompatible with PyCaret in the main environment.

## 1. Configuration

In [ ]:
from pathlib import Path
import sys
import time
import numpy as np
import pandas as pd
import torch

REPO_ROOT = Path('..').resolve()
sys.path.insert(0, str(REPO_ROOT))

from src.data_loader import load_parquet, filter_period, build_series, input_matrix
from src.metrics import all_metrics

PARQUET_PATH = REPO_ROOT / 'data' / 'anonymized_series.parquet'
OUTPUT_DIR = REPO_ROOT / 'outputs' / 'chronos'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

HORIZON = 12
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

## 2. Loading the series

In [ ]:
df = load_parquet(PARQUET_PATH)
df = filter_period(df, 2020, 2024)
series = build_series(df, min_months=24)
train_list, test_list, ids = input_matrix(series, horizon=HORIZON)
print(f'Series: {len(ids)}')

## 3. Loading the Chronos-Bolt model

In [ ]:
from chronos import BaseChronosPipeline

pipeline = BaseChronosPipeline.from_pretrained(
    'amazon/chronos-bolt-small',
    device_map=DEVICE,
    torch_dtype=torch.bfloat16 if DEVICE == 'cuda' else torch.float32,
)

## 4. Per-series inference

Chronos does not support covariates in its current API. Each series is passed as a 1-D tensor.

In [ ]:
forecasts = np.zeros((len(ids), HORIZON), dtype=np.float32)
times = []

for i, train in enumerate(train_list):
    ctx = torch.tensor(train, dtype=torch.float32).unsqueeze(0)
    t0 = time.perf_counter()
    forecast = pipeline.predict(context=ctx, prediction_length=HORIZON)
    dt = time.perf_counter() - t0
    # forecast: (batch, num_quantiles, horizon) — extract the median
    median_idx = forecast.shape[1] // 2
    pred = forecast[0, median_idx, :HORIZON].cpu().numpy()
    forecasts[i] = np.clip(pred, 0, None).astype(np.float32)
    times.append({'series_id': ids[i], 'time_seconds': dt})

pd.DataFrame(times).to_csv(OUTPUT_DIR / 'inference_times_chronos.csv', index=False)

## 5. Evaluation with the 6 metrics

In [ ]:
rows = []
for i, sid in enumerate(ids):
    m = all_metrics(test_list[i], forecasts[i], train_list[i])
    rows.append({'series_id': sid, 'model': 'Chronos_Bolt', **m})

metrics_df = pd.DataFrame(rows)
metrics_df.to_csv(OUTPUT_DIR / 'metrics_chronos.csv', index=False)
metrics_df[['MASE', 'MAE', 'RMSE', 'sMAPE', 'MedAE']].median()

## 6. Export forecasts for later comparison

In [ ]:
pred_rows = []
for i, sid in enumerate(ids):
    for h in range(HORIZON):
        pred_rows.append({
            'series_id': sid,
            'horizon_month': h + 1,
            'y_true': float(test_list[i][h]),
            'pred_chronos': float(forecasts[i, h]),
        })
pd.DataFrame(pred_rows).to_csv(OUTPUT_DIR / 'predictions_chronos.csv', index=False)